# Results and Discussion

## 0. Academic Paper Visualization Standards
For future plots, diagrams, and illustrations in this notebook, please adhere to the following academic standards:
- **Font & Text**: Use clear, legible fonts (e.g., serif for print, sans-serif like Helvetica for presentations). Font sizes should be large enough to be legible when scaled down (typically at least 10pt-12pt).
- **Colors**: Use colorblind-friendly palettes (e.g., Viridis, colorblind-safe seaborn palettes). Ensure high contrast between elements. Use greyscale or patterns where color is not strictly necessary.
- **Axes & Labels**: All axes must be clearly labeled with units. Titles should be concise. Include a legend if multiple data series are plotted.
- **Resolution**: Save all figures as high-resolution images (minimum 300 DPI for PNG/JPEG) or vector graphics (PDF/SVG) for lossless scaling.
- **Layout**: Keep whitespace minimal but sufficient. For side-by-side comparisons, ensure subplots share the same scale if directly comparable, and use clear panel labels (e.g., (a), (b)).

## 1. Environment and Demand

### Arterial Pruning

In [ ]:
import os
import sys
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('.'))
from utils_simplified import *

# PATHS FOR OUTPUTS
BASE_DIR = 'results_and_discussion'
PKL_DIR = os.path.join(BASE_DIR, 'pkl')
IMG_DIR = os.path.join(BASE_DIR, 'images')

os.makedirs(PKL_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

print(f"Pickle files saved in: {PKL_DIR}")
print(f"Images saved in: {IMG_DIR}")

In [ ]:
# Configuration for P1 (Which has the Iligan Graph. All profiles share the same coordinates anyway.)

yaml_p1 = "configs/profile_p1.yaml"
pkl_p1 = os.path.join(PKL_DIR, "profile_p1.pkl")

# Reuse existing pkl if exists, otherwise build.

try:
    cg = reuse_citygraph(pkl_p1)
except FileNotFoundError:
    cg = build_citygraph(yaml_p1, pkl_p1)

print(cg)

In [ ]:
import os
import matplotlib.pyplot as plt

out_path = os.path.join(IMG_DIR, "citygraph_comparison.png")

if os.path.exists(out_path):
    print(f"Generated in previous run already. File found. Now displaying.")

    img = plt.imread(out_path)
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("Rendering whole city graph...")
    img_whole = cg.draw(size=800, only_drivable=False)

    print("Rendering arterial city graph...")
    img_arterial = cg.draw(size=800, only_drivable=True)

    # Plot side by side
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    axes[0].imshow(img_whole)
    axes[0].set_title("(a) Whole City Graph", fontsize=16, pad=15)
    axes[0].axis('off')

    axes[1].imshow(img_arterial)
    axes[1].set_title("(b) Arterial City Graph (Drivable Only)", fontsize=16, pad=15)
    axes[1].axis('off')

    plt.tight_layout()

    # Save the figure
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    print(f"Saved new figure to {out_path}")

    plt.show()


In [ ]:
## CityGraph Arterial Pruning Statistics
import matplotlib.pyplot as plt

total_edges = len(cg.graph)
drivable_edges = sum(1 for e in cg.graph if e.is_drivable)
pruned_edges = total_edges - drivable_edges

print("--- CityGraph Arterial Network Statistics ---")
print(f"Total Network Edges: {total_edges:,}")
print(f"Drivable (Arterial) Edges: {drivable_edges:,} ({(drivable_edges/total_edges)*100:.1f}%)")
print(f"Pruned (Non-drivable) Edges: {pruned_edges:,} ({(pruned_edges/total_edges)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie([drivable_edges, pruned_edges], labels=['Arterials (Drivable)', 'Local/Pruned Edges'], 
       autopct='%1.1f%%', colors=['#2ca02c', '#d62728'], startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 1}, textprops={'fontsize': 12})
ax.set_title("Proportion of Arterial Edges in CityGraph", fontsize=14, pad=15)
plt.tight_layout()

out_path = os.path.join(IMG_DIR, "citygraph_pruning_pie.png")
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()

## 2. Direct Demand Model Topography Analysis

In [ ]:
import datetime
# Target 1 PM (Tomorrow, to ensure valid TomTom future routing queries if necessary)
target_1pm = datetime.datetime.now().replace(hour=13, minute=0, second=0, microsecond=0) + datetime.timedelta(days=1)
ddm_1pm_pkl = os.path.join(PKL_DIR, "ddm_1pm.pkl")

try:
    ddm_1pm = reuse_ddm(ddm_1pm_pkl)
except FileNotFoundError:
    print("Building 1 PM DDM...")
    ddm_1pm = build_ddm(yaml_p1, cg, target_1pm, ddm_1pm_pkl)

print("1 PM DDM Loaded.")

In [ ]:
## Points Pre-Imputed (Empirical Traffic Data)
import numpy as np

img_whole = cg.draw(size=800, only_drivable=False)
min_lat, max_lat, min_lon, max_lon = cg.bbox

# Compute geographically correct figure height
lat_mid   = (min_lat + max_lat) / 2
geo_aspect = (max_lat - min_lat) / ((max_lon - min_lon) * np.cos(np.radians(lat_mid)))
FIG_W = 10
fig, ax = plt.subplots(figsize=(FIG_W, FIG_W * geo_aspect))

# aspect='auto': stretch the image to fill the axes exactly.
ax.imshow(img_whole, extent=[min_lon, max_lon, min_lat, max_lat], aspect='auto')

xs, ys, colors = [], [], []
for node, weight in ddm_1pm.empirical_traffic.items():
    xs.append(node.lon)
    ys.append(node.lat)
    colors.append(weight)

sc = ax.scatter(xs, ys, c=colors, cmap='plasma', s=30, alpha=0.8, edgecolors='none')
plt.colorbar(sc, label="Friction Weight (Travel Time / Free Flow)", shrink=0.8)
ax.set_title("Pre-Imputed Points (TomTom Empirical Data)", fontsize=16, pad=15)
ax.axis('off')
plt.tight_layout()

out_path = os.path.join(IMG_DIR, "ddm_pre_imputed.png")
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
## Betweenness Centrality, IDW Traffic Weights, OD Centrality Map
import numpy as np
import matplotlib.gridspec as gridspec

FIG_W   = 16   # total figure width in inches
row1_h  = (FIG_W / 2) * geo_aspect   # top row height
row2_h  = FIG_W       * geo_aspect   # bottom row height (full width)
fig_h   = row1_h + row2_h

fig = plt.figure(figsize=(FIG_W, fig_h))
gs  = gridspec.GridSpec(
    2, 2, figure=fig,
    height_ratios=[row1_h, row2_h],
    hspace=0.08, wspace=0.05
)

# Subplot 1: Betweenness Centrality
ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(img_whole, extent=[min_lon, max_lon, min_lat, max_lat], aspect='auto', alpha=0.5)
xs, ys, cs = [], [], []
for n, c in ddm_1pm.centrality_scores.items():
    xs.append(n.lon); ys.append(n.lat); cs.append(c)
sc1 = ax1.scatter(xs, ys, c=cs, cmap='Reds', s=10, alpha=0.6, edgecolors='none')
plt.colorbar(sc1, ax=ax1, label="Betweenness Centrality", shrink=0.8)
ax1.set_title("(a) Betweenness Centrality Map", fontsize=16, pad=15)
ax1.axis('off')

# Subplot 2: IDW Traffic Weights
ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(img_whole, extent=[min_lon, max_lon, min_lat, max_lat], aspect='auto', alpha=0.5)
xs, ys, ws = [], [], []
for n, w in ddm_1pm.traffic_weights.items():
    xs.append(n.lon); ys.append(n.lat); ws.append(w)
sc2 = ax2.scatter(xs, ys, c=ws, cmap='Blues', s=10, alpha=0.6, edgecolors='none')
plt.colorbar(sc2, ax=ax2, label="IDW Traffic Weights", shrink=0.8)
ax2.set_title("(b) IDW Traffic Weights Map", fontsize=16, pad=15)
ax2.axis('off')

# Subplot 3: Combined OD Centrality (spans both columns)
ax3 = fig.add_subplot(gs[1, :])
ax3.imshow(img_whole, extent=[min_lon, max_lon, min_lat, max_lat], aspect='auto', alpha=0.5)
xs, ys, ps = [], [], []
for n, p in ddm_1pm.node_probabilities.items():
    xs.append(n.lon); ys.append(n.lat); ps.append(p)
sc3 = ax3.scatter(xs, ys, c=ps, cmap='Purples', s=15, alpha=0.8, edgecolors='none')
plt.colorbar(sc3, ax=ax3, label="OD Centrality Probability", shrink=0.8)
ax3.set_title("(c) Combined OD Centrality Map", fontsize=16, pad=15)
ax3.axis('off')

plt.tight_layout()
out_path = os.path.join(IMG_DIR, "ddm_3maps_comparison.png")
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
## Complete DDM vs Arterials Only DDM
from PIL import Image

context = ((min_lon, max_lat), (max_lon, min_lat))

img_ddm_whole = cg.draw(size=800, only_drivable=False)
ddm_1pm.draw_density(img_ddm_whole, context=context, only_drivable=False)

img_ddm_arterial = cg.draw(size=800, only_drivable=True)
ddm_1pm.draw_density(img_ddm_arterial, context=context, only_drivable=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(img_ddm_whole)
axes[0].set_title("(a) Complete DDM Sampling", fontsize=16, pad=15)
axes[0].axis('off')

axes[1].imshow(img_ddm_arterial)
axes[1].set_title("(b) Arterials-Only DDM Sampling", fontsize=16, pad=15)
axes[1].axis('off')

plt.tight_layout()
out_path = os.path.join(IMG_DIR, "ddm_whole_vs_arterials.png")
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## Time Comparison: 8 AM vs 1 PM vs 5 PM
target_8am = datetime.datetime.now().replace(hour=8, minute=0, second=0, microsecond=0) + datetime.timedelta(days=1)
target_5pm = datetime.datetime.now().replace(hour=17, minute=0, second=0, microsecond=0) + datetime.timedelta(days=1)

ddm_8am_pkl = os.path.join(PKL_DIR, "ddm_8am.pkl")
ddm_5pm_pkl = os.path.join(PKL_DIR, "ddm_5pm.pkl")

from utils_simplified import build_ddm, reuse_ddm

try:
    ddm_8am = reuse_ddm(ddm_8am_pkl)
except FileNotFoundError:
    print("Building 8 AM DDM...")
    ddm_8am = build_ddm(yaml_p1, cg, target_8am, ddm_8am_pkl)

try:
    ddm_5pm = reuse_ddm(ddm_5pm_pkl)
except FileNotFoundError:
    print("Building 5 PM DDM...")
    ddm_5pm = build_ddm(yaml_p1, cg, target_5pm, ddm_5pm_pkl)

img_8am = cg.draw(size=800, only_drivable=True)
ddm_8am.draw_density(img_8am, context=context, only_drivable=True)

img_1pm = cg.draw(size=800, only_drivable=True)
ddm_1pm.draw_density(img_1pm, context=context, only_drivable=True)

img_5pm = cg.draw(size=800, only_drivable=True)
ddm_5pm.draw_density(img_5pm, context=context, only_drivable=True)

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

axes[0].imshow(img_8am)
axes[0].set_title("(a) 8:00 AM Direct Demand Model", fontsize=16, pad=15)
axes[0].axis('off')

axes[1].imshow(img_1pm)
axes[1].set_title("(b) 1:00 PM Direct Demand Model", fontsize=16, pad=15)
axes[1].axis('off')

axes[2].imshow(img_5pm)
axes[2].set_title("(c) 5:00 PM Direct Demand Model", fontsize=16, pad=15)
axes[2].axis('off')

plt.tight_layout()
out_path = os.path.join(IMG_DIR, "ddm_time_comparison.png")
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## DDM Metrics & Distributions (8 AM, 1 PM, 5 PM)
import numpy as np

times = ['8 AM', '1 PM', '5 PM']
ddms = [ddm_8am, ddm_1pm, ddm_5pm]

fig, axes = plt.subplots(3, 3, figsize=(18, 12))

for i, (time_label, ddm) in enumerate(zip(times, ddms)):
    # 1. Betweenness values
    betw_vals = list(ddm.centrality_scores.values())
    axes[i, 0].hist(betw_vals, bins=50, color='indianred', alpha=0.7)
    axes[i, 0].set_title(f"{time_label} - Betweenness Centrality", fontsize=12)
    axes[i, 0].set_yscale('log')  # Log scale since centrality is heavily skewed
    axes[i, 0].set_ylabel("Frequency (Log)")
    if i == 2: axes[i, 0].set_xlabel("Centrality Score")

    # 2. IDW Traffic Weights
    idw_vals = list(ddm.traffic_weights.values())
    axes[i, 1].hist(idw_vals, bins=50, color='steelblue', alpha=0.7)
    axes[i, 1].set_title(f"{time_label} - IDW Traffic Weights", fontsize=12)
    axes[i, 1].set_ylabel("Frequency")
    if i == 2: axes[i, 1].set_xlabel("Friction Weight (TT / FFTT)")

    # 3. OD Centrality (Combined Probabilities)
    od_vals = list(ddm.node_probabilities.values())
    axes[i, 2].hist(od_vals, bins=50, color='mediumpurple', alpha=0.7)
    axes[i, 2].set_title(f"{time_label} - Combined OD Centrality", fontsize=12)
    axes[i, 2].set_yscale('log')
    axes[i, 2].set_ylabel("Frequency (Log)")
    if i == 2: axes[i, 2].set_xlabel("Probability")

plt.tight_layout()
out_path = os.path.join(IMG_DIR, "ddm_distributions.png")
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()

print("\n--- IDW Traffic Weights Statistical Summary ---")
for time_label, ddm in zip(times, ddms):
    idw_vals = list(ddm.traffic_weights.values())
    print(f"{time_label} -> Min: {np.min(idw_vals):.3f} | Mean: {np.mean(idw_vals):.3f} | Max: {np.max(idw_vals):.3f}")
